In [1]:
!pip install -q numpy scikit-learn sentence-transformers

In [2]:
%%writefile support_retriever.py
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
class SupportRetriever:
    def __init__(self, records, backend="tfidf"):
        if not records or any(not r["text"].strip() for r in records):
            raise ValueError("Provide nonempty records")
        self.records=records
        self.backend=backend
        texts=[r["text"] for r in records]
        if backend=="tfidf":
            self.vectorizer=TfidfVectorizer()
            self.matrix=self.vectorizer.fit_transform(texts)
        elif backend=="embedding":
            from sentence_transformers import SentenceTransformer
            self.model=SentenceTransformer(
                "sentence-transformers/all-MiniLM-L6-v2", device="cpu")
            self.check_length(texts)
            self.matrix=self.model.encode(texts, normalize_embeddings=True)
        else:
            raise ValueError("Use tfidf or embedding")
    def check_length(self, texts):
        for text in texts:
            tokens=self.model.tokenizer(text, truncation=False)["input_ids"]
            if len(tokens)>self.model.max_seq_length:
                raise ValueError("Input exceeds encoder token limit")
    def search(self,query,k=3,filters=None):
        if not query.strip() or k<=0:
            return[]
        if self.backend=="tfidf":
            q = self.vectorizer.transform([query])
            if q.nnz==0:
                return[]
            scores=(q @ self.matrix.T).toarray()[0]
        else:
            self.check_length([query])
            q=self.model.encode([query],normalize_embeddings=True)[0]
            scores=self.matrix @ q
        allowed=[
            i for i, r in enumerate(self.records)
            if all(r.get(key)==value for key, value in (filters or {}).items())
        ]
        ranked=sorted(allowed, key=lambda i: (-float(scores[i]), i))
        return[(self.records[i], float(scores[i])) for i in ranked[:k]]
def show_matches(hits):
    if not hits:
        print("No matching records.")
    for record,score in hits:
        print(record["id"], f"score={score:.3f}")
        print(record["text"])

Writing support_retriever.py


In [3]:
%%writefile text_splitter.py
import re
def word_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
def windows(text, size=40, overlap=10, offset=0):
    if size<=0 or overlap<0 or overlap>=size:
        raise ValueError("Invalid size or overlap")
    spans=word_spans(text)
    result=[]
    for i in range(0,len(spans),size-overlap):
        j=min(i+size,len(spans))
        result.append((offset+spans[i][0],offset+spans[j-1][1]))
        if j==len(spans):
            break
    return result
def sentence_chunks(text,size=40):
    if size<=0:
        raise ValueError("Size must be positive")
    cuts=[0]+[m.end() for m in re.finditer(r"[.!?]\s+",text)]+[len(text)]
    result,pending,count=[],None,0
    for a, b in zip(cuts,cuts[1:]):
        n=len(text[a:b].split())
        if not n:
            continue
        if n>size:
            if pending is not None:
                result.append(pending)
            result.extend(windows(text[a:b],size,0,a))
            pending,count=None,0
        elif pending is not None and count+n<=size:
            pending, count=(pending[0], b),count+n
        else:
            if pending is not None:
                result.append(pending)
            pending,count=(a,b),n
    if pending is not None:
        result.append(pending)
    return result
def make_chunks(docs,strategy):
    result=[]
    for doc_id,text in docs.items():
        spans=sentence_chunks(text) if strategy=="sentence"else windows(text, 40, int(strategy))
        for i,(a, b) in enumerate(spans):
            result.append({
                "id":f"{doc_id}:{i}", "doc": doc_id,
                "start":a, "end": b, "text": text[a:b]
            })
    return result
if __name__=="__main__":
    assert windows("")==[]
    assert windows("short text")==[(0, 10)]
    try:
        windows("a b", 2, 2)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid overlap")
    long=" ".join(["word"] * 95)+"."
    assert all(len(long[a:b].split())<=40 for a, b in sentence_chunks(long))
    docs={"example": "Verify your billing details. Keep payment tokens secure."}
    for strategy in["0", "10","sentence"]:
        for c in make_chunks(docs, strategy):
            assert c["text"]==docs[c["doc"]][c["start"]:c["end"]]
            print(strategy,c)

Writing text_splitter.py


In [4]:
%%writefile generate_dataset.py
import json
from pathlib import Path

DOCS = {
    "billing": (
        "Billing and Invoice Policy. Subscription renewals are charged automatically "
        "on the first business day of each calendar month using the saved payment method. "
        "Late payments incur a fixed fee of fifteen dollars after a grace period of three days. "
        "If a credit card declines, notification is sent immediately via email and SMS. "
        "Users must update expired cards through the billing portal before the next cycle. "
        "Refund requests require submitting an official support ticket within seven days of "
        "transaction. Finance team verifies the account activity and issues refunds to the "
        "original bank account within four business days. Corporate accounts follow separate "
        "invoicing terms negotiated during onboarding. Tax exemptions must be verified with "
        "valid documentation uploaded directly to the customer profile."
    ),
    "security": (
        "Security and Access Control. Every account must use a strong passphrase containing "
        "numbers, uppercase letters, and symbols. Multi-factor authentication is mandatory "
        "for admin roles and recommended for standard users. If unusual login locations are "
        "detected, sessions are terminated immediately. Password resets require clicking "
        "the secure token sent to the registered email address. Support agents never ask "
        "for credentials over phone or chat. Inactive sessions automatically log out after "
        "twenty minutes of inactivity to protect sensitive workspace data."
    ),
    "support": (
        "Customer Support Portal Guide. Users can raise tickets for technical assistance "
        "anytime through the web dashboard or mobile application. Response times depend on "
        "subscription tiers, with enterprise users receiving priority assistance under two hours. "
        "Attachments such as error logs and screenshots should be included to expedite troubleshooting. "
        "Live chat support operates between 9 AM and 6 PM standard business hours. Phone callbacks "
        "can be scheduled via the ticketing menu for complex integration issues."
    )
}

evidence = {
    "billing": "Late payments incur a fixed fee of fifteen dollars after a grace period of three days.",
    "security": "Password resets require clicking the secure token sent to the registered email address.",
    "support": "Response times depend on subscription tiers, with enterprise users receiving priority assistance under two hours."
}

questions = [
    ("What happens if a payment is late?", "billing", "dev"),
    ("How do I reset my account password?", "security", "dev"),
    ("What are the support response times?", "support", "dev"),
    ("How are refunds processed?", "billing", "test"),
    ("Are support agents allowed to ask for my password?", "security", "test"),
    ("What is tomorrow's weather forecast?", None, "test")
]

queries = [
    dict(query=q, doc=doc, evidence=evidence[doc] if doc else None, split=split)
    for q, doc, split in questions
]

for d, e in evidence.items():
    assert e in DOCS[d], d
    print(d, len(DOCS[d].split()), "words")

for name, data in [("corpus.json", DOCS), ("queries.json", queries)]:
    if Path(name).exists():
        Path(name).unlink()
    Path(name).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
print("Created corpus.json and queries.json successfully.")

Overwriting generate_dataset.py


In [5]:
!python generate_dataset.py

Created corpus.json and queries.json successfully.


In [6]:
%%writefile run_retrieval_pipeline.py
import csv, json, time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from text_splitter import make_chunks

def evaluate_retrieval(scores, chunks, items):
    hits, reciprocal = [], []
    for row, q in zip(scores, items):
        if q["doc"] is None:
            continue
        top = np.argsort(-row, kind="stable")[:3]
        good = [r for r, i in enumerate(top, 1)
                if chunks[i]["doc"] == q["doc"]
                and q["evidence"] in chunks[i]["text"]]
        hits.append(bool(good))
        reciprocal.append(1/good[0] if good else 0)
    if not hits:
        return None, None
    return float(np.mean(hits)), float(np.mean(reciprocal))

def main():
    with open("corpus.json", encoding="utf-8") as f:
        docs = json.load(f)
    with open("queries.json", encoding="utf-8") as f:
        items = json.load(f)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    queries = [q["query"] for q in items]
    logs = []
    for strategy in ["0", "10", "sentence"]:
        chunks = make_chunks(docs, strategy)
        texts = [c["text"] for c in chunks]
        for method in ["tfidf", "embedding"]:
            if method == "tfidf":
                vec = TfidfVectorizer()
                C = vec.fit_transform(texts)
                Q = vec.transform(queries)
                scores = (Q @ C.T).toarray()
            else:
                C = model.encode(texts, normalize_embeddings=True)
                Q = model.encode(queries, normalize_embeddings=True)
                scores = Q @ C.T
            for split in ["dev", "test"]:
                indices = [i for i, q in enumerate(items) if q["split"] == split]
                res = evaluate_retrieval(scores[indices], chunks, [items[i] for i in indices])
                print(strategy, method, split, "Hit/MRR:", res)
            for row, q in zip(scores, items):
                top = np.argsort(-row, kind="stable")[:3]
                for rank, i in enumerate(top, 1):
                    c = chunks[i]
                    logs.append({
                        "strategy": strategy, "method": method,
                        "query": q["query"], "split": q["split"],
                        "rank": rank, "score": float(row[i]),
                        "chunk_id": c["id"], "doc": c["doc"],
                        "text": c["text"]
                    })
    with open("retrieval_log.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(logs[0]))
        writer.writeheader()
        writer.writerows(logs)
    print("Saved retrieval_log.csv with", len(logs), "rows.")

if __name__ == "__main__":
    main()

Writing run_retrieval_pipeline.py


In [7]:
!python run_retrieval_pipeline.py

Saved retrieval_log.csv with 108 rows.


In [8]:
from google.colab import files
files.download("corpus.json")
files.download("queries.json")
files.download(
    "retrieval_log.csv"
)
print("All 3 files (corpus.json, queries.json, retrieval_log.csv) ready for download.")

All 3 files (corpus.json, queries.json, retrieval_log.csv) ready for download.
